In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ==========================================
# 1. 데이터 불러오기
# ==========================================

df = pd.read_csv("train(6).csv")

print("데이터 크기:", df.shape)
print("\n데이터 정보:")
print(df.info())

# ==========================================
# 2. ID 컬럼 제거
# ==========================================

df = df.drop(columns=["ID"])

# ==========================================
# 3. 입력 데이터(X)와 목표 데이터(y) 분리
# ==========================================

X = df.drop(columns=["Cancer"])
y = df["Cancer"]

print("\nCancer 클래스 분포:")
print(y.value_counts())
print("\nCancer 비율:")
print(y.value_counts(normalize=True))

# ==========================================
# 4. 숫자형 / 범주형 변수 구분
# ==========================================

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object"]
).columns

print("\n숫자형 변수:")
print(list(numeric_features))

print("\n범주형 변수:")
print(list(categorical_features))

# ==========================================
# 5. 범주형 변수 One-Hot Encoding
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

# ==========================================
# 6. Random Forest 모델 설정
# ==========================================

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# ==========================================
# 7. 전처리 + 모델 Pipeline
# ==========================================

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# ==========================================
# 8. Train / Test 데이터 분리
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain 데이터:", X_train.shape)
print("Test 데이터 :", X_test.shape)

# ==========================================
# 9. 모델 학습
# ==========================================

print("\n모델 학습 시작...")

pipeline.fit(X_train, y_train)

print("모델 학습 완료!")

# ==========================================
# 10. Test 데이터 예측
# ==========================================

y_pred = pipeline.predict(X_test)

# ==========================================
# 11. 평가 지표 계산
# ==========================================

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n========================================")
print("모델 평가 결과")
print("========================================")

print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1 Score : {f1:.4f} ({f1*100:.2f}%)")

# ==========================================
# 12. Classification Report
# ==========================================

print("\n========================================")
print("Classification Report")
print("========================================")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Normal", "Cancer"]
    )
)

# ==========================================
# 13. Confusion Matrix 계산
# ==========================================

cm = confusion_matrix(y_test, y_pred)

print("\n========================================")
print("Confusion Matrix")
print("========================================")

print(cm)

# ==========================================
# 14. Confusion Matrix의 각 값 출력
# ==========================================

TN, FP, FN, TP = cm.ravel()

print("\n----------------------------------------")
print("Confusion Matrix 상세")
print("----------------------------------------")

print(f"TN (정상 → 정상): {TN}")
print(f"FP (정상 → 암)  : {FP}")
print(f"FN (암 → 정상)  : {FN}")
print(f"TP (암 → 암)    : {TP}")

# ==========================================
# 15. Confusion Matrix 시각화
# ==========================================

plt.figure(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal (0)", "Cancer (1)"],
    yticklabels=["Normal (0)", "Cancer (1)"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Random Forest")

plt.tight_layout()
plt.show()